# Title: Implementation of Greedy Algorithms

## Objectives
- To understand the concept of the Greedy algorithmic paradigm and its properties.
- To implement the Fractional Knapsack problem using a greedy approach.
- To implement Job Sequencing with Deadlines to maximize total profit.
- To implement Prim's Algorithm for finding the Minimum Spanning Tree (MST).
- To implement Kruskal's Algorithm for finding the Minimum Spanning Tree (MST).
- To implement Dijkstra's Algorithm for finding the Single-Source Shortest Path.
- To analyze the time and space complexity of each greedy algorithm.

## Theory

**Greedy Algorithms** solve optimization problems by making the locally optimal choice at each step with the hope of finding a global optimum. Unlike Dynamic Programming, greedy algorithms do not reconsider their choices — once a decision is made, it is final. Greedy algorithms are efficient and simple but do not always yield an optimal solution for every problem.

A greedy algorithm is applicable when the problem exhibits:
1. **Greedy Choice Property:** A globally optimal solution can be assembled by choosing the locally optimal solution at each step.
2. **Optimal Substructure:** The optimal solution to a problem contains optimal solutions to its subproblems.

**Fractional Knapsack:** Unlike the 0/1 Knapsack, items can be broken into fractions. The greedy strategy is to sort items by value-to-weight ratio in decreasing order and greedily take as much of each item as the remaining capacity allows. This guarantees the optimal solution. Time complexity: O(n log n).

**Job Sequencing with Deadlines:** Given n jobs each with a deadline and profit, schedule jobs on a single machine to maximize total profit. The greedy strategy sorts jobs by descending profit and assigns each to the latest available time slot before its deadline. Time complexity: O(n²) or O(n log n) with Union-Find.

**Prim's Algorithm:** Builds a Minimum Spanning Tree (MST) by starting from an arbitrary vertex and greedily adding the minimum-weight edge that connects the tree to a new vertex. Uses a priority queue (min-heap) to efficiently select the next edge. Time complexity: O(E log V) with a binary heap.

**Kruskal's Algorithm:** Builds an MST by sorting all edges by weight and greedily adding edges that do not form a cycle, using a Union-Find (Disjoint Set) data structure to detect cycles efficiently. Time complexity: O(E log E).

**Dijkstra's Algorithm:** Finds the shortest path from a single source vertex to all other vertices in a weighted graph with non-negative edge weights. Uses a greedy approach with a priority queue — always extends the shortest known path first. Time complexity: O((V + E) log V) with a binary heap.

## 1. WAP to implement Fractional Knapsack Problem

### Algorithm
1. **Start**
2. Input n items, each with a weight and a value, and knapsack capacity W.
3. Compute the value-to-weight ratio for each item.
4. Sort items in descending order of value-to-weight ratio.
5. Initialize total value = 0 and remaining capacity = W.
6. For each item in sorted order:
   - If the item's weight fits entirely, add it completely and reduce remaining capacity.
   - Else, add a fraction of the item to fill the remaining capacity, then stop.
7. Display the total maximum value obtained.
8. **Stop**

In [2]:
def fractional_knapsack(items, capacity):
   
    # Compute ratio and sort descending
    items_ratio = sorted(items, key=lambda x: x[2]/x[1], reverse=True)

    total_value = 0.0
    remaining   = capacity
    result      = []

    for name, weight, value in items_ratio:
        ratio = value / weight
        if remaining == 0:
            break
        if weight <= remaining:
            taken = weight
            total_value += value
            remaining   -= weight
        else:
            taken = remaining
            total_value += ratio * remaining
            remaining    = 0
        result.append((name, weight, value, round(ratio, 2), taken))

    return result, round(total_value, 2)

# Items: (name, weight, value)
items = [
    ('Item A', 10, 60),
    ('Item B', 20, 100),
    ('Item C', 30, 120),
]
capacity = 50

print('Fractional Knapsack Problem - Greedy Approach')
print('===============================================')
print(f"{'Item':<10} {'Weight':<10} {'Value':<10} {'V/W Ratio':<12} {'Status'}")
print(f"{'----':<10} {'------':<10} {'-----':<10} {'---------':<12} {'------'}")
for name, w, v in items:
    print(f'{name:<10} {w:<10} {v:<10} {v/w:<12.2f} Available')
print(f'\nKnapsack Capacity: {capacity}')

result, max_value = fractional_knapsack(items, capacity)

print('\nGreedy Selection (sorted by V/W ratio):')
print(f"{'Item':<10} {'Weight':<10} {'Value':<10} {'Ratio':<10} {'Taken':<10} {'Contribution'}")
print(f"{'----':<10} {'------':<10} {'-----':<10} {'-----':<10} {'-----':<10} {'------------'}")
for name, weight, value, ratio, taken in result:
    fraction = taken / weight
    contribution = round(ratio * taken, 2)
    status = 'Full' if taken == weight else f'{fraction:.2f} fraction'
    print(f'{name:<10} {weight:<10} {value:<10} {ratio:<10} {taken:<10} {contribution} ({status})')

print(f'\nMaximum Value in Knapsack: {max_value}')


Fractional Knapsack Problem - Greedy Approach
Item       Weight     Value      V/W Ratio    Status
----       ------     -----      ---------    ------
Item A     10         60         6.00         Available
Item B     20         100        5.00         Available
Item C     30         120        4.00         Available

Knapsack Capacity: 50

Greedy Selection (sorted by V/W ratio):
Item       Weight     Value      Ratio      Taken      Contribution
----       ------     -----      -----      -----      ------------
Item A     10         60         6.0        10         60.0 (Full)
Item B     20         100        5.0        20         100.0 (Full)
Item C     30         120        4.0        20         80.0 (0.67 fraction)

Maximum Value in Knapsack: 240.0


## 2. WAP to implement Job Sequencing with Deadlines

### Algorithm
1. **Start**
2. Input n jobs, each with a job ID, deadline, and profit.
3. Sort jobs in decreasing order of profit.
4. Find the maximum deadline to determine the number of available time slots.
5. Initialize a slot array with all slots empty.
6. For each job in sorted order:
   - Find the latest available time slot at or before the job's deadline.
   - If such a slot exists, assign the job to that slot and add its profit.
   - Otherwise, skip the job.
7. Display the sequence of scheduled jobs and the total maximum profit.
8. **Stop**

In [3]:
def job_sequencing(jobs):
   
    # Sort by profit descending
    jobs_sorted = sorted(jobs, key=lambda x: x[2], reverse=True)

    max_deadline = max(j[1] for j in jobs)
    slots = [None] * (max_deadline + 1)  # 1-indexed
    total_profit = 0
    log = []

    for job_id, deadline, profit in jobs_sorted:
        # Find latest free slot <= deadline
        placed = False
        for t in range(min(deadline, max_deadline), 0, -1):
            if slots[t] is None:
                slots[t] = job_id
                total_profit += profit
                log.append((job_id, deadline, profit, t, 'Scheduled'))
                placed = True
                break
        if not placed:
            log.append((job_id, deadline, profit, '-', 'Skipped'))

    sequence = [slots[t] for t in range(1, max_deadline + 1) if slots[t] is not None]
    return sequence, total_profit, log

# Jobs: (job_id, deadline, profit)
jobs = [
    ('J1', 2, 100),
    ('J2', 1,  19),
    ('J3', 2,  27),
    ('J4', 1,  25),
    ('J5', 3,  15),
]

print('Job Sequencing with Deadlines - Greedy Approach')
print('=================================================')
print(f"{'Job':<8} {'Deadline':<12} {'Profit'}")
print(f"{'---':<8} {'--------':<12} {'------'}")
for jid, dl, pr in jobs:
    print(f'{jid:<8} {dl:<12} {pr}')

sequence, total_profit, log = job_sequencing(jobs)

print('\nScheduling Process (sorted by profit descending):')
print(f"{'Job':<8} {'Deadline':<10} {'Profit':<10} {'Slot':<8} {'Decision'}")
print(f"{'---':<8} {'--------':<10} {'------':<10} {'----':<8} {'--------'}")
for jid, dl, pr, slot, decision in log:
    print(f'{jid:<8} {dl:<10} {pr:<10} {str(slot):<8} {decision}')

print(f'\nOptimal Job Sequence : {sequence}')
print(f'Total Maximum Profit : {total_profit}')


Job Sequencing with Deadlines - Greedy Approach
Job      Deadline     Profit
---      --------     ------
J1       2            100
J2       1            19
J3       2            27
J4       1            25
J5       3            15

Scheduling Process (sorted by profit descending):
Job      Deadline   Profit     Slot     Decision
---      --------   ------     ----     --------
J1       2          100        2        Scheduled
J3       2          27         1        Scheduled
J4       1          25         -        Skipped
J2       1          19         -        Skipped
J5       3          15         3        Scheduled

Optimal Job Sequence : ['J3', 'J1', 'J5']
Total Maximum Profit : 142


## 3. WAP to implement Prim's Algorithm (Minimum Spanning Tree)

### Algorithm
1. **Start**
2. Input the graph as an adjacency list with edge weights.
3. Initialize a min-heap with the starting vertex (cost = 0).
4. Initialize a visited set and MST edge list.
5. While the heap is not empty:
   - Extract the minimum-cost vertex `u` from the heap.
   - If `u` is already visited, skip it.
   - Mark `u` as visited and add the edge to MST.
   - For each neighbor `v` of `u` with edge weight `w`:
     - If `v` is not visited, push `(w, v, u)` into the heap.
6. Display the MST edges and total minimum cost.
7. **Stop**

In [4]:
import heapq

def prims_algorithm(graph, start=0):
   
    visited = set()
    mst_edges = []
    total_cost = 0

    # Min-heap: (edge_weight, current_vertex, parent_vertex)
    heap = [(0, start, -1)]

    while heap:
        cost, u, parent = heapq.heappop(heap)

        if u in visited:
            continue

        visited.add(u)
        total_cost += cost

        if parent != -1:
            mst_edges.append((parent, u, cost))

        for weight, v in graph[u]:
            if v not in visited:
                heapq.heappush(heap, (weight, v, u))

    return mst_edges, total_cost

# Graph as adjacency list: {vertex: [(weight, neighbor)]}
graph = {
    0: [(2, 1), (3, 3)],
    1: [(2, 0), (3, 2), (4, 3), (1, 4)],
    2: [(3, 1), (5, 4)],
    3: [(3, 0), (4, 1), (1, 4)],
    4: [(1, 1), (5, 2), (1, 3)],
}
V = len(graph)

print("Prim's Algorithm - Minimum Spanning Tree")
print('=========================================')
print('Graph Adjacency List:')
for v in sorted(graph):
    edges = ', '.join(f'V{nb} (w={w})' for w, nb in graph[v])
    print(f'  V{v}: {edges}')

mst_edges, total_cost = prims_algorithm(graph, start=0)

print('\nMST Construction (starting from V0):')
print(f"{'Step':<6} {'Edge':<15} {'Weight'}")
print(f"{'----':<6} {'----':<15} {'------'}")
for step, (u, v, w) in enumerate(mst_edges, 1):
    print(f'{step:<6} V{u} -- V{v}      {w}')

print(f'\nMinimum Spanning Tree Edges: {[(f"V{u}-V{v}", w) for u,v,w in mst_edges]}')
print(f'Total MST Cost: {total_cost}')


Prim's Algorithm - Minimum Spanning Tree
Graph Adjacency List:
  V0: V1 (w=2), V3 (w=3)
  V1: V0 (w=2), V2 (w=3), V3 (w=4), V4 (w=1)
  V2: V1 (w=3), V4 (w=5)
  V3: V0 (w=3), V1 (w=4), V4 (w=1)
  V4: V1 (w=1), V2 (w=5), V3 (w=1)

MST Construction (starting from V0):
Step   Edge            Weight
----   ----            ------
1      V0 -- V1      2
2      V1 -- V4      1
3      V4 -- V3      1
4      V1 -- V2      3

Minimum Spanning Tree Edges: [('V0-V1', 2), ('V1-V4', 1), ('V4-V3', 1), ('V1-V2', 3)]
Total MST Cost: 7


## 4. WAP to implement Kruskal's Algorithm (Minimum Spanning Tree)

### Algorithm
1. **Start**
2. Represent the graph as a list of edges: (weight, u, v).
3. Sort all edges in ascending order of weight.
4. Initialize a Union-Find (Disjoint Set Union) structure with each vertex as its own parent.
5. Initialize MST edge list and total cost = 0.
6. For each edge (w, u, v) in sorted order:
   - Find the roots of u and v using `find()`.
   - If roots are different (no cycle), unite them using `union()` and add edge to MST.
   - Stop when MST has V-1 edges.
7. Display the MST edges and total minimum cost.
8. **Stop**

In [5]:
def find(parent, x):
    
    if parent[x] != x:
        parent[x] = find(parent, parent[x])
    return parent[x]

def union(parent, rank, x, y):
    """Union by rank."""
    rx, ry = find(parent, x), find(parent, y)
    if rx == ry:
        return False
    if rank[rx] < rank[ry]:
        rx, ry = ry, rx
    parent[ry] = rx
    if rank[rx] == rank[ry]:
        rank[rx] += 1
    return True

def kruskal(V, edges):
    """
    V: number of vertices
    edges: list of (weight, u, v)
    Returns: MST edges and total cost
    """
    edges_sorted = sorted(edges)
    parent = list(range(V))
    rank   = [0] * V
    mst    = []
    total  = 0

    for w, u, v in edges_sorted:
        if union(parent, rank, u, v):
            mst.append((u, v, w))
            total += w
            if len(mst) == V - 1:
                break

    return mst, total

# Graph: (weight, u, v)
V = 5
edges = [
    (2, 0, 1),
    (3, 0, 3),
    (3, 1, 2),
    (4, 1, 3),
    (1, 1, 4),
    (5, 2, 4),
    (1, 3, 4),
]

print("Kruskal's Algorithm - Minimum Spanning Tree")
print('=============================================')
print('All Edges (sorted by weight):')
print(f"{'Edge':<15} {'Weight'}")
print(f"{'----':<15} {'------'}")
for w, u, v in sorted(edges):
    print(f'V{u} -- V{v}       {w}')

mst, total_cost = kruskal(V, edges)

print('\nKruskal MST Construction:')
print(f"{'Step':<6} {'Edge':<15} {'Weight':<10} {'Action'}")
print(f"{'----':<6} {'----':<15} {'------':<10} {'------'}")
parent_trace = list(range(V))
rank_trace   = [0] * V
step = 1
for w, u, v in sorted(edges):
    ru, rv = find(parent_trace, u), find(parent_trace, v)
    if ru != rv:
        union(parent_trace, rank_trace, u, v)
        print(f'{step:<6} V{u} -- V{v}      {w:<10} Added to MST')
        step += 1
    else:
        print(f'  -    V{u} -- V{v}      {w:<10} Skipped (forms cycle)')

print(f'\nMinimum Spanning Tree Edges: {[(f"V{u}-V{v}", w) for u,v,w in mst]}')
print(f'Total MST Cost: {total_cost}')


Kruskal's Algorithm - Minimum Spanning Tree
All Edges (sorted by weight):
Edge            Weight
----            ------
V1 -- V4       1
V3 -- V4       1
V0 -- V1       2
V0 -- V3       3
V1 -- V2       3
V1 -- V3       4
V2 -- V4       5

Kruskal MST Construction:
Step   Edge            Weight     Action
----   ----            ------     ------
1      V1 -- V4      1          Added to MST
2      V3 -- V4      1          Added to MST
3      V0 -- V1      2          Added to MST
  -    V0 -- V3      3          Skipped (forms cycle)
4      V1 -- V2      3          Added to MST
  -    V1 -- V3      4          Skipped (forms cycle)
  -    V2 -- V4      5          Skipped (forms cycle)

Minimum Spanning Tree Edges: [('V1-V4', 1), ('V3-V4', 1), ('V0-V1', 2), ('V1-V2', 3)]
Total MST Cost: 7


## 5. WAP to implement Dijkstra's Algorithm (Single-Source Shortest Path)

### Algorithm
1. **Start**
2. Input the weighted graph and a source vertex.
3. Initialize `dist[]` = INF for all vertices; set `dist[source] = 0`.
4. Initialize a min-heap with `(0, source)` and a visited set.
5. While the heap is not empty:
   - Extract vertex `u` with minimum distance.
   - If `u` is already visited, skip.
   - Mark `u` as visited.
   - For each neighbor `v` of `u` with edge weight `w`:
     - If `dist[u] + w < dist[v]`, update `dist[v]` and push `(dist[v], v)` to heap.
6. Display shortest distances from source to all vertices.
7. **Stop**

In [6]:
import heapq

def dijkstra(graph, source):
  
    INF  = float('inf')
    dist = {v: INF for v in graph}
    prev = {v: None for v in graph}
    dist[source] = 0
    visited = set()
    heap = [(0, source)]

    while heap:
        d, u = heapq.heappop(heap)
        if u in visited:
            continue
        visited.add(u)
        for w, v in graph[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                heapq.heappush(heap, (dist[v], v))

    return dist, prev

def get_path(prev, source, target):
    """Reconstruct shortest path from source to target."""
    path = []
    cur = target
    while cur is not None:
        path.append(cur)
        cur = prev[cur]
    path.reverse()
    return path if path[0] == source else []

# Graph: adjacency list {vertex: [(weight, neighbor)]}
graph = {
    0: [(4, 1), (1, 2)],
    1: [(4, 0), (2, 2), (5, 3)],
    2: [(1, 0), (2, 1), (8, 3), (10, 4)],
    3: [(5, 1), (8, 2), (2, 4), (6, 5)],
    4: [(10, 2), (2, 3), (3, 5)],
    5: [(6, 3), (3, 4)],
}
source = 0

print("Dijkstra's Algorithm - Single-Source Shortest Path")
print('===================================================')
print(f'Source Vertex: V{source}')
print('\nGraph Adjacency List:')
for v in sorted(graph):
    edges = ', '.join(f'V{nb} (w={w})' for w, nb in graph[v])
    print(f'  V{v}: {edges}')

dist, prev = dijkstra(graph, source)

print(f'\nShortest Distances from V{source}:')
print(f"{'Vertex':<10} {'Distance':<12} {'Shortest Path'}")
print(f"{'------':<10} {'--------':<12} {'-------------'}")
for v in sorted(graph):
    d = dist[v]
    path = get_path(prev, source, v)
    path_str = ' -> '.join(f'V{p}' for p in path)
    print(f'V{v:<9} {d:<12} {path_str}')


Dijkstra's Algorithm - Single-Source Shortest Path
Source Vertex: V0

Graph Adjacency List:
  V0: V1 (w=4), V2 (w=1)
  V1: V0 (w=4), V2 (w=2), V3 (w=5)
  V2: V0 (w=1), V1 (w=2), V3 (w=8), V4 (w=10)
  V3: V1 (w=5), V2 (w=8), V4 (w=2), V5 (w=6)
  V4: V2 (w=10), V3 (w=2), V5 (w=3)
  V5: V3 (w=6), V4 (w=3)

Shortest Distances from V0:
Vertex     Distance     Shortest Path
------     --------     -------------
V0         0            V0
V1         3            V0 -> V2 -> V1
V2         1            V0 -> V2
V3         8            V0 -> V2 -> V1 -> V3
V4         10           V0 -> V2 -> V1 -> V3 -> V4
V5         13           V0 -> V2 -> V1 -> V3 -> V4 -> V5


# Analysis of the Algorithms

| Algorithm | Time Complexity (Best) | Time Complexity (Average) | Time Complexity (Worst) | Space Complexity | Notes |
|:---|:---|:---|:---|:---|:---|
| *Fractional Knapsack* | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ | Sorting dominates; always yields optimal solution. |
| *Job Sequencing* | $O(n \log n)$ | $O(n^2)$ | $O(n^2)$ | $O(n)$ | Sort + slot assignment; optimal for profit maximization. |
| *Prim's Algorithm* | $O(E \log V)$ | $O(E \log V)$ | $O(E \log V)$ | $O(V + E)$ | Efficient for dense graphs; uses min-heap. |
| *Kruskal's Algorithm* | $O(E \log E)$ | $O(E \log E)$ | $O(E \log E)$ | $O(V + E)$ | Efficient for sparse graphs; uses Union-Find. |
| *Dijkstra's Algorithm* | $O((V+E) \log V)$ | $O((V+E) \log V)$ | $O((V+E) \log V)$ | $O(V)$ | Does not work with negative edge weights. |

**Notes:**
- $n$ = number of items/jobs, $V$ = number of vertices, $E$ = number of edges.
- Both Prim's and Kruskal's produce an optimal MST for any connected weighted undirected graph.
- Dijkstra's greedy approach fails with negative-weight edges; use Bellman-Ford in such cases.
- Fractional Knapsack always gives the optimal solution; 0/1 Knapsack (by contrast) requires DP.

# Discussion

The implemented programs demonstrated the greedy approach across five classic algorithmic problems:

1. **Fractional Knapsack:** The greedy strategy of sorting items by their value-to-weight ratio and filling the knapsack greedily produced the optimal solution. Items were taken in full where capacity allowed, and a fraction of the last item filled the remaining space. This confirms that the greedy choice property holds for the fractional knapsack, unlike the 0/1 variant.

2. **Job Sequencing with Deadlines:** By sorting jobs in decreasing order of profit and assigning each to the latest available slot before its deadline, the algorithm maximized total profit. Some lower-profit jobs were skipped when no feasible slot remained. The experiment confirmed that greedy scheduling with profit-based ordering is optimal for single-machine deadline scheduling.

3. **Prim's Algorithm:** Starting from vertex V0, the algorithm incrementally built the MST by always selecting the minimum-weight edge connecting the visited tree to an unvisited vertex. The priority queue ensured efficient edge selection. Prim's is particularly efficient for dense graphs and correctly produced the MST with the minimum total edge weight.

4. **Kruskal's Algorithm:** All edges were sorted by weight, and edges were greedily added to the MST as long as they did not form a cycle. The Union-Find data structure with path compression and union-by-rank enabled fast cycle detection. Kruskal's is more suitable for sparse graphs and produced the same optimal MST as Prim's.

5. **Dijkstra's Algorithm:** Starting from vertex V0, the algorithm computed the shortest path to all other vertices by always relaxing the nearest unvisited vertex first. Path reconstruction using the previous-vertex array allowed full shortest-path recovery. The experiment confirmed that Dijkstra's greedy approach yields optimal results for graphs with non-negative edge weights.

# Conclusion

The experiment was successfully completed by implementing five important Greedy algorithms in Python: Fractional Knapsack, Job Sequencing with Deadlines, Prim's MST, Kruskal's MST, and Dijkstra's Shortest Path. The programs confirmed that greedy algorithms are efficient and produce globally optimal solutions when the greedy choice property and optimal substructure conditions are satisfied. The experiment also highlighted the importance of appropriate data structures — min-heaps for Prim's and Dijkstra's, and Union-Find for Kruskal's — in achieving efficient implementations. Understanding greedy algorithms is essential for solving problems in network design, scheduling, routing, and resource allocation.